In [5]:
import pandas as pd

In [8]:
train= pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
train

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [9]:
test.isnull().sum() + train.isnull().sum()

Age             263.0
Cabin          1014.0
Embarked          2.0
Fare              1.0
Name              0.0
Parch             0.0
PassengerId       0.0
Pclass            0.0
Sex               0.0
SibSp             0.0
Survived          NaN
Ticket            0.0
dtype: float64

In [11]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

for col in ['Age', 'Fare']:
    imputer = SimpleImputer(strategy='median')
    train[col] = imputer.fit_transform(train[[col]])
    test[col] = imputer.transform(test[[col]])

train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])
test['Embarked'] = test['Embarked'].fillna(train['Embarked'].mode()[0])

for col in ['Sex', 'Embarked']:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X = train[features]
y = train['Survived']
X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=42)

In [12]:
test.isnull().sum() + train.isnull().sum()

Age               0.0
Cabin          1014.0
Embarked          0.0
Fare              0.0
Name              0.0
Parch             0.0
PassengerId       0.0
Pclass            0.0
Sex               0.0
SibSp             0.0
Survived          NaN
Ticket            0.0
dtype: float64

In [13]:
classifiers_grid = {
    'LogisticRegression': (
        LogisticRegression(max_iter=1000),
        {'C': [0.01, 0.1, 1, 10, 100]}
    ),
    'DecisionTree': (
        DecisionTreeClassifier(),
        {'max_depth': [3, 5, 7, 10, None],
         'min_samples_split': [2, 5, 10]}
    ),
    'RandomForest': (
        RandomForestClassifier(),
        {'n_estimators': [50, 100, 200],
         'max_depth': [3, 5, 7, None],
         'min_samples_split': [2, 5]}
    ),
    'GradientBoosting': (
        GradientBoostingClassifier(),
        {'n_estimators': [50, 100, 200],
         'max_depth': [3, 5, 7],
         'learning_rate': [0.05, 0.1, 0.2]}
    ),
    'XGBoost': (
        XGBClassifier(eval_metric='logloss'),
        {'n_estimators': [50, 100, 200],
         'max_depth': [3, 5, 7],
         'learning_rate': [0.05, 0.1, 0.2]}
    ),
}

best_score = 0
best_name = ''
best_estimator = None

for name, (clf, params) in classifiers_grid.items():
    grid = GridSearchCV(clf, param_grid=params, cv=4, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)
    
    val_score = grid.score(X_val, y_val)
    print(f"{name}:")
    print(f"  best params : {grid.best_params_}")
    print(f"  CV accuracy : {grid.best_score_:.4f}")
    print(f"  val accuracy: {val_score:.4f}")
    
    if val_score > best_score:
        best_score = val_score
        best_name = name
        best_estimator = grid.best_estimator_

print(f"\nBest clf: {best_name} (val accuracy={best_score:.4f})")

best_estimator.fit(X, y)
predictions = best_estimator.predict(test[features])

LogisticRegression:
  best params : {'C': 1}
  CV accuracy : 0.8009
  val accuracy: 0.8072
DecisionTree:
  best params : {'max_depth': 3, 'min_samples_split': 2}
  CV accuracy : 0.8204
  val accuracy: 0.8027
RandomForest:
  best params : {'max_depth': 7, 'min_samples_split': 5, 'n_estimators': 50}
  CV accuracy : 0.8353
  val accuracy: 0.8296
GradientBoosting:
  best params : {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200}
  CV accuracy : 0.8249
  val accuracy: 0.8027
XGBoost:
  best params : {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
  CV accuracy : 0.8323
  val accuracy: 0.8206

Best clf: RandomForest (val accuracy=0.8296)


In [14]:
output = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})
output.to_csv('submission.csv', index=False)